<a href="https://colab.research.google.com/github/MonaKhadija/Fundamentals-of-Predictive-Analysis-Machine-Learning-and-AI-Assignments/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Assignment 3: Feature Engineering**

Student name: Khadija bassiouni

1. setting up the environment and loading the "House Prices - Advanced Regression Techniques" dataset

In [2]:
import pandas as pd
import numpy as np

# Loading the dataset
df = pd.read_csv('train.csv')

# Displaying the first 10 rows
display(df.head(10))

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000


**2. Handling Missing Values:**

In [3]:
#  Inspecting the dataset for missing values
missing_values = df.isnull().sum()
missing_values_filtered = missing_values[missing_values > 0]
print("Missing values per column:")
print(missing_values_filtered)

Missing values per column:
LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


In [9]:
# 2. Identify numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 3. Impute missing values in numerical columns using the median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

# Impute missing values in categorical columns using the most frequent category
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        most_frequent_val = df[col].mode()[0]
        df[col] = df[col].fillna(most_frequent_val)

print(f"Total Numerical Columns Found: {len(num_cols)}")
print(list(num_cols))  #  numerical column names

print(f"\nTotal Categorical Columns Found: {len(cat_cols)}")
print(list(cat_cols))  #  categorical column names

print("Total remaining missing values across the entire dataset:", df.isnull().sum().sum())

Total Numerical Columns Found: 40
['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice', 'TotalSF', 'Age']

Total Categorical Columns Found: 43
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Ele

**3. Feature Scaling and Encoding Categorical Features:**

In [10]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Feature Scaling

# Identifying numerical features
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'Id' in num_features:
    num_features.remove('Id')
if 'SalePrice' in num_features:
    num_features.remove('SalePrice')

print(f"Identified {len(num_features)} numerical features for scaling.")

# Scaling the numerical features using Min-Max scaling
scaler = MinMaxScaler()
df[num_features] = scaler.fit_transform(df[num_features])

# Viewing the first few rows of scaled numerical features
display(df[num_features].head())


# Encoding Categorical Features

# Identifying categorical features
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nIdentified {len(cat_features)} categorical features for encoding.")

# Applying One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_array = encoder.fit_transform(df[cat_features])

# Converting the encoded array back into a DataFrame with proper column names
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(cat_features), index=df.index)

# Dropping original categorical columns and concatenating the new one-hot encoded columns
df = df.drop(columns=cat_features)
df = pd.concat([df, encoded_df], axis=1)

print(f"\nDataset shape after One-Hot Encoding: {df.shape}")

Identified 38 numerical features for scaling.


,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,TotalSF,Age
0,0.235294,0.150685,0.033420,0.666667,0.500,0.949275,0.883333,0.12250,0.125089,0.0,...,0.111517,0.000000,0.0,0.0,0.0,0.0,0.090909,0.50,0.195481,0.036765
1,0.000000,0.202055,0.038795,0.555556,0.875,0.753623,0.433333,0.00000,0.173281,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.363636,0.25,0.191802,0.227941
2,0.235294,0.160959,0.046507,0.666667,0.500,0.934783,0.866667,0.10125,0.086109,0.0,...,0.076782,0.000000,0.0,0.0,0.0,0.0,0.727273,0.50,0.207742,0.051471
3,0.294118,0.133562,0.038561,0.666667,0.500,0.311594,0.333333,0.00000,0.038271,0.0,...,0.063985,0.492754,0.0,0.0,0.0,0.0,0.090909,0.00,0.187336,0.669118
4,0.235294,0.215753,0.060576,0.777778,0.500,0.927536,0.833333,0.21875,0.116052,0.0,...,0.153565,0.000000,0.0,0.0,0.0,0.0,1.000000,0.50,0.263531,0.058824



Identified 43 categorical features for encoding.

Dataset shape after One-Hot Encoding: (1460, 291)


**4. Feature Engineering:**

In [11]:
# TotalSF combining basement and floor square footages
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Creating Age by subtracting YearBuilt from YrSold
df['Age'] = df['YrSold'] - df['YearBuilt']

# Separating features (X) and target variable (y), dropping 'Id' and 'SalePrice'
X = df.drop(columns=['SalePrice', 'Id'])
y = df['SalePrice']

**5. Data Leakage Prevention:**

In [12]:
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (1168, 289)
Testing set shape: (292, 289)


To prevent data leakage we must split the data before applying any transformations.

What to do to prevent data leakage: All preprocessing parameters (such as the median for numerical imputation, most frequent category for categorical imputation, Min-Max scaling bounds, and One-Hot categories) should be calculated only on the training set and then applied to the test set using a Pipeline or ColumnTransformer.

**6. Train a simple linear regression:**

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Identifigerical ad categorical cols fro X_train
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# numerical Pipeline  Imputation by media -> Normalisation Min-Max
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

# Categorical Pipeline : Imputation by most frequent value using One-Hot encoding
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combined process
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# create and train model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFi...
                                                   'BsmtHalfBath', 'FullBath',
                                                   'HalfBath', 'BedroomAbvGr',
                                                   'KitchenAbvGr',
                                                   'TotRmsAbvGrd', 'Fireplaces',
                                                   'GarageYrBlt', 'GarageCars',
                                                   'GarageArea', 'WoodDeckSF',
                                                   'OpenPorchSF',
                                                   'EnclosedPorch', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  [])])),
                ('regressor', LinearRegression())])

In [19]:
from sklearn.metrics import mean_squared_error

# compute predictions on test and train sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

#  Mean Squared Error (MSE)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Training Mean Squared Error (MSE): {train_mse:,.2f}")
print(f"Test Mean Squared Error (MSE): {test_mse:,.2f}")

Training Mean Squared Error (MSE): 382,336,962.33
Test Mean Squared Error (MSE): 873,658,179.21


**Insights:**

The training MSE is pretty low, which makes sense since the model fits the training data closely. Also, when we check the test set, the MSE rises up massively and reaches an very high number.
Doing One-Hot Encoding on categorical columns with lots of unique values  created hundreds of new binary columns, making the data very sparse. Because we didn't use any regularization, the standard linear regression ran into major issues with multicollinearity and coefficient variance. This caused numerical instability, meaning the model memorized the training data instead of learning patterns that could generalize to the test set.